# PySpark Feature Engineering Pipeline
**Goal:** Use PySpark to load, clean, join, and feature-engineer the raw telemetry data into a ML-ready dataset.

### Why PySpark?
The telemetry dataset has **876,000 rows** of time-series data across 100 machines. PySpark's distributed DataFrame API with **Window functions** makes rolling-average feature engineering on this scale both efficient and scalable to real production fleet sizes (millions of vehicles).

In [1]:
import os, sys

os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')

# ← ADD THESE TWO LINES:
os.environ['PYSPARK_PYTHON'] = sys.executable         # e.g. d:\...\ml-venv\Scripts\python.exe
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

print(f'PYSPARK_PYTHON set to: {sys.executable}')


PYSPARK_PYTHON set to: d:\Programming\Projects\ml-venv\Scripts\python.exe


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import pandas as pd

spark = SparkSession.builder \
    .appName('FleetTelemetryPipeline') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'Spark version: {spark.version}')
print('Spark session initialized ✅')

Spark version: 3.5.0
Spark session initialized ✅


## 1. Load Raw CSVs into Spark DataFrames

In [3]:
DATA_DIR = '../data/raw/'

telemetry_spark = spark.read.csv(DATA_DIR + 'PdM_telemetry.csv', header=True, inferSchema=True)
failures_spark  = spark.read.csv(DATA_DIR + 'PdM_failures.csv',  header=True, inferSchema=True)
errors_spark    = spark.read.csv(DATA_DIR + 'PdM_errors.csv',    header=True, inferSchema=True)
machines_spark  = spark.read.csv(DATA_DIR + 'PdM_machines.csv',  header=True, inferSchema=True)
maint_spark     = spark.read.csv(DATA_DIR + 'PdM_maint.csv',     header=True, inferSchema=True)

print('Dataset sizes:')
for name, df in [('telemetry', telemetry_spark), ('failures', failures_spark),
                 ('errors', errors_spark), ('machines', machines_spark), ('maint', maint_spark)]:
    print(f'  {name:12s}: {df.count():>8,} rows  |  {len(df.columns)} cols')

Dataset sizes:
  telemetry   :  876,100 rows  |  6 cols
  failures    :      761 rows  |  3 cols
  errors      :    3,919 rows  |  3 cols
  machines    :      100 rows  |  3 cols
  maint       :    3,286 rows  |  3 cols


In [4]:
# Cast datetime strings to proper timestamp type
telemetry_spark = telemetry_spark.withColumn('datetime', F.to_timestamp('datetime'))
failures_spark  = failures_spark.withColumn('datetime',  F.to_timestamp('datetime'))
errors_spark    = errors_spark.withColumn('datetime',    F.to_timestamp('datetime'))
maint_spark     = maint_spark.withColumn('datetime',     F.to_timestamp('datetime'))

print('Schema after timestamp cast:')
telemetry_spark.printSchema()

Schema after timestamp cast:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)



## 2. Data Quality Checks

In [5]:
print('=== Null Value Audit ===')
for col in telemetry_spark.columns:
    null_count = telemetry_spark.filter(F.col(col).isNull()).count()
    print(f'  {col:12s}: {null_count} nulls')

print('\n=== Telemetry Date Range ===')
telemetry_spark.select(
    F.min('datetime').alias('start'),
    F.max('datetime').alias('end'),
    F.countDistinct('machineID').alias('unique_machines')
).show()

=== Null Value Audit ===


  datetime    : 0 nulls
  machineID   : 0 nulls
  volt        : 0 nulls
  rotate      : 0 nulls
  pressure    : 0 nulls
  vibration   : 0 nulls

=== Telemetry Date Range ===
+-------------------+-------------------+---------------+
|              start|                end|unique_machines|
+-------------------+-------------------+---------------+
|2015-01-01 06:00:00|2016-01-01 06:00:00|            100|
+-------------------+-------------------+---------------+



## 3. Feature Engineering — Rolling Window Statistics

For each machine, we compute **3-hour** and **24-hour** rolling averages and standard deviations of each sensor reading. These rolling features are the core signal for predicting upcoming failures — sudden spikes or drift in rolling averages are strong indicators of component degradation.

In [ ]:
# Convert datetime to unix timestamp for window calculations
telemetry_spark = telemetry_spark.withColumn('ts', F.unix_timestamp('datetime'))

sensors = ['volt', 'rotate', 'pressure', 'vibration']

# Window specs: 3h = 3*3600 seconds, 24h = 24*3600 seconds
#WindowSpec as a blueprint or a set of instructions.
w3h  = Window.partitionBy('machineID').orderBy('ts').rangeBetween(-3*3600, 0)
w24h = Window.partitionBy('machineID').orderBy('ts').rangeBetween(-24*3600, 0)

tel_featured = telemetry_spark

for sensor in sensors:
    # 3-hour rolling mean and std
    tel_featured = tel_featured \
        .withColumn(f'{sensor}_mean3h',  F.avg(sensor).over(w3h)) \
        .withColumn(f'{sensor}_std3h',   F.stddev(sensor).over(w3h)) \
        .withColumn(f'{sensor}_mean24h', F.avg(sensor).over(w24h)) \
        .withColumn(f'{sensor}_std24h',  F.stddev(sensor).over(w24h))

print(f'Feature columns after rolling windows: {len(tel_featured.columns)}')
tel_featured.select('machineID', 'datetime', 'volt', 'volt_mean3h', 'volt_std3h', 'volt_mean24h').show(5)

Feature columns after rolling windows: 23
+---------+-------------------+----------------+------------------+------------------+------------------+
|machineID|           datetime|            volt|       volt_mean3h|        volt_std3h|      volt_mean24h|
+---------+-------------------+----------------+------------------+------------------+------------------+
|       12|2015-01-01 06:00:00|171.404215158513|  171.404215158513|              NULL|  171.404215158513|
|       12|2015-01-01 07:00:00|177.897737109469|  174.650976133991| 4.591613405304684|  174.650976133991|
|       12|2015-01-01 08:00:00|183.402122783571|177.56802501718434| 6.005745531797465|177.56802501718434|
|       12|2015-01-01 09:00:00|176.204974071792|177.22726228083624| 4.950804301498617|177.22726228083624|
|       12|2015-01-01 10:00:00|148.871381245529|171.59405380259022|15.456900833795691| 171.5560860737748|
+---------+-------------------+----------------+------------------+------------------+------------------+
only

## 4. Error Count Features
Aggregate the error logs to count how many errors each machine logged in the last 24h.

In [7]:
# One-hot encode error types
error_ids = [row['errorID'] for row in errors_spark.select('errorID').distinct().orderBy('errorID').collect()]
print(f'Error types: {error_ids}')

errors_pivot = errors_spark \
    .withColumn('ts', F.unix_timestamp('datetime'))

for eid in error_ids:
    errors_pivot = errors_pivot.withColumn(f'is_{eid}', (F.col('errorID') == eid).cast('int'))

# Aggregate error counts per machine per hour to join with telemetry
errors_agg = errors_pivot.groupBy('machineID', 'datetime').agg(
    *[F.sum(f'is_{eid}').alias(f'error_{eid}_count') for eid in error_ids]
)

# Left join errors onto telemetry
tel_featured = tel_featured.join(errors_agg, on=['machineID', 'datetime'], how='left')
for eid in error_ids:
    tel_featured = tel_featured.withColumn(f'error_{eid}_count', 
                                           F.coalesce(F.col(f'error_{eid}_count'), F.lit(0)))

print('Error features joined ✅')

Error types: ['error1', 'error2', 'error3', 'error4', 'error5']
Error features joined ✅


## 5. Fleet Metadata Join
Attach machine model type and age to each telemetry row.

In [8]:
# Index-encode the model string
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol='model', outputCol='model_idx')
machines_indexed = indexer.fit(machines_spark).transform(machines_spark)

tel_featured = tel_featured.join(machines_indexed.select('machineID', 'model_idx', 'age'), 
                                  on='machineID', how='left')

print('Machine metadata joined ✅')
tel_featured.select('machineID', 'datetime', 'model_idx', 'age', 'volt_mean24h').show(3)

Machine metadata joined ✅
+---------+-------------------+---------+---+------------------+
|machineID|           datetime|model_idx|age|      volt_mean24h|
+---------+-------------------+---------+---+------------------+
|       12|2015-01-01 06:00:00|      0.0|  9|  171.404215158513|
|       12|2015-01-01 07:00:00|      0.0|  9|  174.650976133991|
|       12|2015-01-01 08:00:00|      0.0|  9|177.56802501718434|
+---------+-------------------+---------+---+------------------+
only showing top 3 rows



## 6. Create Target Label — Failure in Next 24 Hours

In [9]:
# Pure Spark SQL label creation — zero Python UDFs, runs entirely on JVM
failures_ts = failures_spark \
    .withColumn('fail_ts', F.unix_timestamp('datetime')) \
    .select(
        F.col('machineID').alias('f_machineID'),
        F.col('fail_ts')
    )

# Broadcast the tiny failures table (761 rows) to all executors
# Range join: find any failure within 24h after each telemetry reading
tel_labeled = tel_featured.join(
    F.broadcast(failures_ts),
    (tel_featured['machineID'] == F.col('f_machineID')) &
    (F.col('fail_ts') >  tel_featured['ts']) &
    (F.col('fail_ts') <= tel_featured['ts'] + 86400),
    how='left'
).groupBy(
    *[tel_featured[c] for c in tel_featured.columns]
).agg(
    F.when(F.count('fail_ts') > 0, F.lit(1)).otherwise(F.lit(0)).alias('label')
)

pos   = tel_labeled.filter(F.col('label') == 1).count()
total = tel_labeled.count()
print(f'Positive (failure) samples : {pos:,} ({100*pos/total:.2f}%)')
print(f'Negative (normal) samples  : {total-pos:,} ({100*(total-pos)/total:.2f}%)')


Positive (failure) samples : 17,184 (1.96%)
Negative (normal) samples  : 858,916 (98.04%)


## 7. Drop Nulls & Save as Parquet

In [10]:
std_cols = [f'{s}_std24h' for s in sensors]
tel_clean = tel_labeled.dropna(subset=std_cols).drop('datetime')

print(f'Rows after null drop: {tel_clean.count():,}')
print(f'Feature count: {len(tel_clean.columns)} columns')

OUTPUT_PATH = '../data/processed/fleet_processed.parquet'
tel_clean.write.mode('overwrite').parquet(OUTPUT_PATH)
print(f'\nSaved to: {OUTPUT_PATH} ✅')


Rows after null drop: 876,000
Feature count: 30 columns

Saved to: ../data/processed/fleet_processed.parquet ✅


In [11]:
# Final feature summary
print('=== Final Feature Set ===')
for col in tel_clean.columns:
    print(f'  {col}')

=== Final Feature Set ===
  machineID
  volt
  rotate
  pressure
  vibration
  ts
  volt_mean3h
  volt_std3h
  volt_mean24h
  volt_std24h
  rotate_mean3h
  rotate_std3h
  rotate_mean24h
  rotate_std24h
  pressure_mean3h
  pressure_std3h
  pressure_mean24h
  pressure_std24h
  vibration_mean3h
  vibration_std3h
  vibration_mean24h
  vibration_std24h
  error_error1_count
  error_error2_count
  error_error3_count
  error_error4_count
  error_error5_count
  model_idx
  age
  label


## Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read 5 CSVs into Spark DataFrames | 876,000 telemetry rows |
| Quality | Null audit, timestamp casting | Zero nulls, correct dtypes |
| Windows | 3h + 24h rolling mean & std per sensor | +16 engineered features |
| Errors | One-hot error counts joined per hour | +5 error indicator features |
| Metadata | Model type + vehicle age joined | +2 fleet features |
| Label | Binary failure-within-24h target | ~2% positive class |
| Save | Parquet format | Optimized for columnar ML reads |